In [44]:
import numpy as np
import pandas as pd

In [45]:
class CustomLogisticRegression:
    def __init__(self, learning_rate=0.01, epochs=2500):
        self.lr = learning_rate
        self.epochs = epochs
        self.weights = None
        self.bias = None

    def _sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0.0

        for _ in range(self.epochs):
            z = np.dot(X, self.weights) + self.bias
            y_pred = self._sigmoid(z)


            dw = (1 / n_samples) * np.dot(X.T, (y_pred - y))
            db = (1 / n_samples) * np.sum(y_pred - y)

            self.weights -= self.lr * dw
            self.bias -= self.lr * db

    def predict(self, X, threshold=0.5):
        z = np.dot(X, self.weights) + self.bias
        y_prob = self._sigmoid(z)
        return (y_prob >= threshold).astype(int)

In [46]:
# data load
df = pd.read_csv('/content/car_price_dataset_medium.csv')
df.columns = df.columns.str.strip()

if 'Car_ID' in df.columns:
    df = df.drop(columns=['Car_ID'])

df = df.dropna().drop_duplicates()

median_price = df['Price_USD'].median()
median_mileage = df['Kilometers_Driven'].median()

cond_price = (df['Price_USD'] <= median_price)
cond_mileage = (df[mileage_col] <= median_mileage)

df['Buy_Decision'] = np.where(cond_price & cond_mileage, 1, 0)

df = df.drop(columns=['Price_USD'])

In [47]:
# Categorical Columns Encoding (Label Encoding)
for col in df.select_dtypes(include=['object']).columns:
    unique_vals = df[col].unique()
    mapping = {val: idx for idx, val in enumerate(unique_vals)}
    df[col] = df[col].map(mapping)

# Feature (X) and Target (y) seperate
X = df.drop(columns=['Buy_Decision']).values
y = df['Buy_Decision'].values

In [48]:
# Manual Train-Test Split (80-20 Split)
np.random.seed(42)
indices = np.arange(len(y))
np.random.shuffle(indices)

split_idx = int(0.8 * len(indices))
train_idx, test_idx = indices[:split_idx], indices[split_idx:]

X_train_raw, X_test_raw = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

In [49]:
# Feature Scaling / Standardization
mean = np.mean(X_train_raw, axis=0)
std = np.std(X_train_raw, axis=0)
std[std == 0] = 1.0

X_train = (X_train_raw - mean) / std
X_test = (X_test_raw - mean) / std

In [50]:
#model train
model = CustomLogisticRegression(learning_rate=0.01, epochs=2500)
model.fit(X_train, y_train)


In [51]:
# prediction
y_pred = model.predict(X_test)

In [52]:
# custom evaluation matrix
tp = np.sum((y_test == 1) & (y_pred == 1))
tn = np.sum((y_test == 0) & (y_pred == 0))
fp = np.sum((y_test == 0) & (y_pred == 1))
fn = np.sum((y_test == 1) & (y_pred == 0))

accuracy = (tp + tn) / len(y_test) if len(y_test) > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

In [53]:
#output
print("     CARWISE EVALUATION RESULT   ")
print("======================================")
print(f"Accuracy  : {accuracy * 100:.2f}%")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print("--------------------------------------")
print("Confusion Matrix:")
print(f" TN: {tn:<4} FP: {fp}")
print(f" FN: {fn:<4} TP: {tp}")

     CARWISE EVALUATION RESULT   
Accuracy  : 74.00%
Precision : 0.6000
Recall    : 0.4839
F1-Score  : 0.5357
--------------------------------------
Confusion Matrix:
 TN: 118  FP: 20
 FN: 32   TP: 30
